In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from copy import deepcopy
from functools import partial
from itertools import combinations
import random
import gc

# Import sklearn classes for model selection, cross validation, and performance evaluation
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error, mean_squared_log_error
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
from sklearn import ensemble
import seaborn as sns
from category_encoders import OneHotEncoder, OrdinalEncoder, CountEncoder, CatBoostEncoder
from imblearn.under_sampling import RandomUnderSampler

# Import libraries for Hypertuning
import optuna
import shap
from sklearn.inspection import permutation_importance

# Import libraries for gradient boosting
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from catboost import CatBoost, CatBoostRegressor, CatBoostClassifier
from catboost import Pool
import torch

# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## Data

In [ ]:
def fix_columns(df): 
    """Removes (in millions) and (approx).1 from names of columns."""
    df.columns = df.columns.str.replace('(in millions)', '', regex=False)
    df.columns = df.columns.str.replace(' home(approx).1', '_home', regex=False)
    return df

filepath = '/kaggle/input/playground-series-s3e11'

df_train = pd.read_csv(os.path.join(filepath, 'train.csv'), index_col=[0])
df_test = pd.read_csv(os.path.join(filepath, 'test.csv'), index_col=[0])
# original = pd.read_csv('/kaggle/input/media-campaign-cost-prediction/train_dataset.csv')

df_train = fix_columns(df_train)
df_test = fix_columns(df_test)
# original = fix_columns(original)

df_train['is_generated'] = 1
df_test['is_generated'] = 1
# original['is_generated'] = 0

# original = original.reset_index()
# original['id'] = original['index'] + df_test.index[-1] + 1
# original = original.drop(columns = ['index']).set_index('id')

target_col = 'cost'

## EDA

In [ ]:
n_cols = 2
n_rows = (len(df_test.columns) - 1) // n_cols + 1

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(20, 30))

for i, var_name in enumerate(df_test.columns.tolist()):
    row = i // n_cols
    col = i % n_cols
    
    ax = axes[row, col]
    sns.distplot(df_train[var_name], kde=True, ax=ax, label='Train')
    sns.distplot(df_test[var_name], kde=True, ax=ax, label='Test')
    # sns.distplot(original[var_name], kde=True, ax=ax, label='Original')
    ax.set_title(f'{var_name} Distribution (Train vs Test)')
    ax.legend()
    
plt.tight_layout()
plt.show()

In [ ]:
def plot_heatmap(df, title):
    # Create a mask for the diagonal elements
    mask = np.zeros_like(df.astype(float).corr())
    mask[np.triu_indices_from(mask)] = True

    # Set the colormap and figure size
    colormap = plt.cm.RdBu_r
    plt.figure(figsize=(15, 15))

    # Set the title and font properties
    plt.title(f'{title} Correlation of Features', fontweight='bold', y=1.02, size=20)

    # Plot the heatmap with the masked diagonal elements
    sns.heatmap(df.astype(float).corr(), linewidths=0.1, vmax=1.0, vmin=-1.0, 
                square=True, cmap=colormap, linecolor='white', annot=True, annot_kws={"size": 8, "weight": "bold"},
                mask=mask)

plot_heatmap(df_train.drop('is_generated', axis=1), title='Train data')
plot_heatmap(df_test.drop('is_generated', axis=1), title='Test data')
# plot_heatmap(original.drop('is_generated', axis=1), title='original')

In [ ]:
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

def hierarchical_clustering(data, title):
    fig, ax = plt.subplots(1, 1, figsize=(14, 8), dpi=120)
    correlations = data.corr()
    converted_corr = 1 - np.abs(correlations)
    Z = linkage(squareform(converted_corr), 'complete')
    
    dn = dendrogram(Z, labels=data.columns, ax=ax, above_threshold_color='#ff0000', orientation='right')
    hierarchy.set_link_color_palette(None)
    plt.grid(axis='x')
    plt.title(f'{title} Hierarchical clustering, Dendrogram', fontsize=18, fontweight='bold')
    plt.show()

hierarchical_clustering(df_train.drop(['is_generated', target_col], axis=1), title='Train data')
hierarchical_clustering(df_test.drop('is_generated', axis=1), title='Test data')

## K-means cluster
This analysis will explain how to determine the number of clusters. When using the silhouette score, it is important to note that the silhouette score is a measure of separation, so the silhouette score is high when the number of clusters is low. Therefore, we also check the silhouette plot and confirm that the thickness of each cluster is similar and that the leading edge of the silhouette score exceeds the leading edge of the silhouette score.

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.cm as cm

def plot_silhouette_analysis(X, range_n_clusters, n_splits=10):
    for n_clusters in range_n_clusters:
        fig, (ax1, ax2) = plt.subplots(1, 2)
        fig.set_size_inches(18, 7)

        # Set limits for the silhouette plot
        ax1.set_xlim([-0.1, 1])
        ax1.set_ylim([0, len(X) // n_splits + (n_clusters + 1) * 10])

        # Initialize the clusterer
        clusterer = MiniBatchKMeans(n_clusters=n_clusters, random_state=0)
        cluster_labels = clusterer.fit_predict(X)

        # Reduce calculation speed
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        _, val_index = next(skf.split(X, cluster_labels))
        cluster_labels_ = cluster_labels[val_index]

        # Compute the silhouette score
        silhouette_avg = silhouette_score(X.iloc[val_index], cluster_labels_)

        # Compute the silhouette scores for each sample
        sample_silhouette_values = silhouette_samples(X.iloc[val_index], cluster_labels_)

        y_lower = 10
        for i in range(n_clusters):
            # Aggregate the silhouette scores for samples belonging to cluster i and sort them
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels_ == i]
            ith_cluster_silhouette_values.sort()

            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = cm.nipy_spectral(float(i) / n_clusters)
            ax1.fill_betweenx(
                np.arange(y_lower, y_upper),
                0,
                ith_cluster_silhouette_values,
                facecolor=color,
                edgecolor=color,
                alpha=0.7,
            )

            # Label the silhouette plots with their cluster numbers at the middle
            ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            y_lower = y_upper + 10  # Update y_lower for the next plot

        # Set titles and labels for the silhouette plot
        ax1.set_title("The silhouette plot for the various clusters.")
        ax1.set_xlabel("The silhouette coefficient values")
        ax1.set_ylabel("Cluster label")
        ax1.axvline(x=silhouette_avg, color="red", linestyle="--")  # Vertical line for avg silhouette score
        ax1.set_yticks([])  # Clear y-axis labels/ticks
        ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

        # Visualize the actual clusters formed
        colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
        ax2.scatter(
            X.iloc[:, 0], X.iloc[:, 1], marker=".", s=30, lw=0, alpha=0.7, c=colors, edgecolor="k"
        )

        # Label the clusters
        centers = clusterer.cluster_centers_
        ax2.scatter(
            centers[:, 0],
            centers[:, 1],
            marker="o",
            c="white",
            alpha=1,
            s=200,
            edgecolor="k",
        )
        
        for i, c in enumerate(centers):
            ax2.scatter(c[0], c[1], marker="$%d$" % i, alpha=1, s=50, edgecolor="k")

        ax2.set_title("The visualization of the clustered data.")
        ax2.set_xlabel("Feature space for the 1st feature")
        ax2.set_ylabel("Feature space for the 2nd feature")

        plt.suptitle(
            f"KMeans clustering with n_clusters = {n_clusters}, silhouette_avg_score = {silhouette_avg:.3f}, cols = {cols}",
            fontsize=14,
            fontweight="bold",
        )

    plt.show()


n_splits = 10
range_n_clusters = [3, 5]
# cols = df_test.select_dtypes(include=['float']).columns.tolist()

cols = ['store_sales', 'gross_weight']
plot_silhouette_analysis(df_train[cols], range_n_clusters, n_splits=n_splits)

cols = ['store_sales', 'units_per_case']
plot_silhouette_analysis(df_train[cols], range_n_clusters, n_splits=n_splits)

## Prepare train and test sets

In [ ]:
def fe(df):
    # kudos to https://www.kaggle.com/code/sergiosaharovskiy/ps-s3e11-2023-eda-and-submission
    df.unit_sales = df.unit_sales.clip(0, 5)
    df['children_ratio'] = df['total_children'] / df['num_children_at_home']
    df['children_ratio'] = df['children_ratio'].replace([np.inf, -np.inf], 10)
    df.fillna(0, inplace = True)
    df['facilities'] = df.eval('coffee_bar + video_store + salad_bar + florist')
    df['independent_child'] = df.eval('total_children - num_children_at_home')
    return df

# Apply FE
df_train = fe(df_train)
# original = fe(original)
df_test = fe(df_test)

# Concatenate train and original dataframes, and prepare train and test sets
df_train = pd.concat([df_train, ])
X_train = df_train.drop([f'{target_col}'],axis=1).reset_index(drop=True)
y_train = df_train[f'{target_col}'].reset_index(drop=True)
X_test = df_test.reset_index(drop=True)

# Transform the values in y_train using the natural logarithm function
y_train_ori = y_train.copy()
y_train = np.log1p(y_train_ori)

# Drop cols
# cols_to_drop = [
#     'store_sales', 
#     'gross_weight', 
#     'unit_sales', 
#     'low_fat',
#     'recyclable_package', 
#     'salad_bar', 
#     'units_per_case'
# ]
cols_to_drop = [
    'low_fat', 
    'gross_weight', 
    'recyclable_package', 
    'store_sales', 
    'units_per_case', 
    'unit_sales', 
    'prepared_food'
]
X_train.drop(cols_to_drop, axis=1, inplace=True)
X_test.drop(cols_to_drop, axis=1, inplace=True)
X_test_fillna = X_test.fillna(-999)

print(f"X_train shape :{X_train.shape} , y_train shape :{y_train.shape}")
print(f"X_test shape :{X_test.shape}")

# Delete the train and test dataframes to free up memory
del df_train, df_test

In [ ]:
categorical_columns = [
    'total_children', 
    'num_children_at_home', 
    'avg_cars_at_home', 
    'store_sqft', 
    'coffee_bar', 
    'video_store', 
#     'prepared_food'
]

for col in categorical_columns:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)

In [ ]:
class Splitter:
    def __init__(self, test_size=0.2, kfold=True, n_splits=5):
        self.test_size = test_size
        self.kfold = kfold
        self.n_splits = n_splits

    def split_data(self, X, y, random_state_list):
        if self.kfold:
            for random_state in random_state_list:
                kf = KFold(n_splits=self.n_splits, random_state=random_state, shuffle=True)
                for train_index, val_index in kf.split(X, y):
                    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
                    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
                    yield X_train, X_val, y_train, y_val
        else:
            for random_state in random_state_list:
                X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=self.test_size, random_state=random_state)
                yield X_train, X_val, y_train, y_val

kfold = True
n_splits = 1 if not kfold else 7
random_state = 42
random_state_list = [42] # used by split_data [42, 41, 35]
n_estimators = 666 # 9999
early_stopping_rounds = 100
verbose = False
device = "gpu" if torch.cuda.is_available() else "cpu"

splitter = Splitter(kfold=kfold, n_splits=n_splits)

## Define Model

In [ ]:
class Regressor:
    def __init__(self, n_estimators=100, device="cpu", random_state=0):
        self.n_estimators = n_estimators
        self.device = device
        self.random_state = random_state
        self.reg_models = self._define_reg_model()
        self.len_models = len(self.reg_models)
        
    def _define_reg_model(self):
        
        xgb_params = {
            'n_estimators': self.n_estimators,
            'learning_rate': 0.05,
            'max_depth': 8,
            'subsample': 1.0,
            'colsample_bytree': 1.0,
            'n_jobs': -1,
            'objective': 'reg:squarederror',
            'verbosity': 0,
            'eval_metric': 'rmse',
            'random_state': self.random_state,
        }
        if self.device == 'gpu':
            xgb_params['tree_method'] = 'gpu_hist'
            xgb_params['predictor'] = 'gpu_predictor'
        
        lgb_params = {
            'n_estimators': self.n_estimators,
            'max_depth': 8,
            'learning_rate': 0.05293702575527996,
            'subsample': 0.20851841295589477,
            'colsample_bytree': 0.5784778854092203,
            'reg_alpha': 0.2622912287429849,
            'reg_lambda': 2.8702494234117617e-08,
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': self.device,
            'random_state': self.random_state
        }
        
        cb_params = {
            'iterations': self.n_estimators,
            'depth': 7,
            'learning_rate': 0.12947105266151432,
            'l2_leaf_reg': 0.6169164517797081,
            'random_strength': 0.21235850198764036,
            'max_bin': 212,
            'od_wait': 67,
            'one_hot_max_size': 73,
            'grow_policy': 'Depthwise',
            'bootstrap_type': 'Bayesian',
            'od_type': 'Iter',
            'loss_function': 'RMSE',
            'task_type': self.device.upper(),
            'random_state': self.random_state
        }
        
#         add_models = {
#             'rf_reg': RandomForestRegressor(n_estimators=1500, max_depth=15, random_state=self.random_state, n_jobs=-1),
#             'hgbc_reg': HistGradientBoostingRegressor(max_iter=2500, max_depth=15, random_state=self.random_state),
#         }
        
        base_reg_models = {
            'xgb_reg': xgb.XGBRegressor(**xgb_params),
            'lgb_reg': lgb.LGBMRegressor(**lgb_params),
            'cat_reg': CatBoostRegressor(**cb_params)
        }
        
        reg_models = {
#             **add_models,
            **base_reg_models
        }
        
        return reg_models

## Weighted Ensemble Model by Optuna on Training
A weighted average is performed during training;  
The weights were determined for each model using the predictions for the train data created in the out of fold with Optuna's CMAsampler. (Here it is defined by `OptunaWeights`)  
This is an extension of the averaging method. All models are assigned different weights defining the importance of each model for prediction.

![](https://www.analyticsvidhya.com/wp-content/uploads/2015/08/Screen-Shot-2015-08-22-at-6.40.37-pm.png)

In [ ]:
class OptunaWeights:
    def __init__(self, random_state):
        self.study = None
        self.weights = None
        self.random_state = random_state

    def _objective(self, trial, y_true, y_preds):
        # Define the weights for the predictions from each model
        weights = [trial.suggest_float(f"weight{n}", 0, 1) for n in range(len(y_preds))]

        # Calculate the weighted prediction
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=weights)

        # Calculate the RMSLE score for the weighted prediction
        score = np.sqrt(mean_squared_log_error(y_true, weighted_pred))
        return score

    def fit(self, y_true, y_preds, n_trials=300):
        optuna.logging.set_verbosity(optuna.logging.ERROR)
        sampler = optuna.samplers.CmaEsSampler(seed=self.random_state)
        self.study = optuna.create_study(sampler=sampler, study_name="OptunaWeights", direction='minimize')
        objective_partial = partial(self._objective, y_true=y_true, y_preds=y_preds)
        self.study.optimize(objective_partial, n_trials=n_trials)
        self.weights = [self.study.best_params[f"weight{n}"] for n in range(len(y_preds))]

    def predict(self, y_preds):
        assert self.weights is not None, 'OptunaWeights error, must be fitted before predict'
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=self.weights)
        return weighted_pred

    def fit_predict(self, y_true, y_preds, n_trials=300):
        self.fit(y_true, y_preds, n_trials=n_trials)
        return self.predict(y_preds)
    
    def weights(self):
        return self.weights

## Train Model

In [ ]:
# Initialize an array for storing test predictions
test_predss = np.zeros(X_test.shape[0])
ensemble_score = []
weights = []
trained_models = dict(zip(Regressor().reg_models.keys(), [[] for _ in range(Regressor().len_models)]))

# Evaluate on validation data and store predictions on test data
for i, (X_train_, X_val, y_train_, y_val) in enumerate(splitter.split_data(X_train, y_train, random_state_list=random_state_list)):
    n = i % n_splits
    m = i // n_splits
        
    # Get a set of Regressor models
    reg = Regressor(n_estimators, device, random_state)
    models = reg.reg_models
    
    # Initialize lists to store oof and test predictions for each base model
    oof_preds = []
    test_preds = []
    
    # Loop over each base model and fit it to the training data, evaluate on validation data, and store predictions
    for name, model in models.items():
        # Train
        if name == 'cat_reg':
            train_pool = Pool(X_train_, y_train_, cat_features=categorical_columns)
            test_pool = Pool(X_val, y_val ,cat_features=categorical_columns)
            model.fit(train_pool, eval_set=[test_pool], early_stopping_rounds=early_stopping_rounds, verbose=verbose)
        elif name == 'lgb_reg':
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)], 
                      categorical_feature=categorical_columns, early_stopping_rounds=early_stopping_rounds, verbose=verbose)
        elif name == 'xgb_reg':
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)], early_stopping_rounds=early_stopping_rounds, verbose=verbose)
        elif name == 'hgbc_reg':
            model.fit(X_train_, y_train_)
        else:
            X_train_ = X_train_.fillna(-999)
            X_val = X_val.fillna(-999)
            model.fit(X_train_, y_train_)
        y_val_pred = model.predict(X_val)
        
        # Predict
        if name in ['rf_reg']:
            test_pred = model.predict(X_test_fillna)
        else:
            test_pred = model.predict(X_test)
        
        # Convert predicted values back to their original scale by applying the expm1 function
        y_val_pred = np.expm1(y_val_pred)
        test_pred = np.expm1(test_pred)
        
        score = np.sqrt(mean_squared_log_error(np.expm1(y_val), y_val_pred))
        print(f'{name} [FOLD-{n} SEED-{random_state_list[m]}] RMSLE score: {score:.5f}')
        
        oof_preds.append(y_val_pred)
        test_preds.append(test_pred)
        trained_models[f'{name}'].append(deepcopy(model))
    
    # Use Optuna to find the best ensemble weights
    y_val = np.expm1(y_val)
    optweights = OptunaWeights(random_state=random_state)
    y_val_pred = optweights.fit_predict(y_val.values, oof_preds)
    score = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
    print(f'Ensemble [FOLD-{n} SEED-{random_state_list[m]}] RMSLE score {score:.5f}')
    ensemble_score.append(score)
    weights.append(optweights.weights)
    test_predss += optweights.predict(test_preds) / (n_splits * len(random_state_list))
    
    gc.collect()

In [ ]:
# Calculate the mean LogLoss score of the ensemble
mean_score = np.mean(ensemble_score)
std_score = np.std(ensemble_score)
print(f'Ensemble RMSLE score {mean_score:.5f} ± {std_score:.5f}')

# Print the mean and standard deviation of the ensemble weights for each model
print('--- Model Weights ---')
mean_weights = np.mean(weights, axis=0)
std_weights = np.std(weights, axis=0)
for name, mean_weight, std_weight in zip(models.keys(), mean_weights, std_weights):
    print(f'{name} {mean_weight:.5f} ± {std_weight:.5f}')

In [ ]:
def visualize_importance(models, feature_cols, title, top=16):
    importances = []
    feature_importance = pd.DataFrame()
    for i, model in enumerate(models):
        _df = pd.DataFrame()
        _df["importance"] = model.feature_importances_
        _df["feature"] = pd.Series(feature_cols)
        _df["fold"] = i
        _df = _df.sort_values('importance', ascending=False)
        _df = _df.head(top)
        feature_importance = pd.concat([feature_importance, _df], axis=0, ignore_index=True)
        
    feature_importance = feature_importance.sort_values('importance', ascending=False)
    # display(feature_importance.groupby(["feature"]).mean().reset_index().drop('fold', axis=1))
    plt.figure(figsize=(16, 10))
    sns.barplot(x='importance', y='feature', data=feature_importance, color='skyblue', errorbar='sd')
    plt.xlabel('Importance', fontsize=14)
    plt.ylabel('Feature', fontsize=14)
    plt.title(f'{title} Feature Importance [Top {top}]', fontsize=18)
    plt.grid(True, axis='x')
    plt.show()
    
for name, models in trained_models.items():
    if name != 'hgbc_reg':
        visualize_importance(models, list(X_train.columns), name)

## SHAP and Permutation importance and Null Importances

In [ ]:
def plot_permutation_importance(model, X, y):
    perm_importance = permutation_importance(model, X, y, n_repeats=2, random_state=random_state, n_jobs=-1)
    sorted_importances_idx = perm_importance.importances_mean.argsort()[::-1]

    importances = pd.DataFrame(
        perm_importance.importances[sorted_importances_idx].T,
        columns=X.columns[sorted_importances_idx],
    ).T

    fig, ax = plt.subplots(figsize=(12, 8))
    colors = ['#00BFFF' if importance >= 0 else '#FF69B4' for importance in perm_importance.importances_mean[sorted_importances_idx]]

    # Plotting the bar graph with error bars
    sns.barplot(x=importances[0], y=importances.index, palette=colors)
    plt.errorbar(x=importances[0], y=importances.index,
                 xerr=2 * perm_importance.importances_std[sorted_importances_idx][::-1], fmt='none',
                 ecolor='black', elinewidth=2, capsize=8)

    plt.xlabel('Permutation Importance')
    plt.tight_layout()
    plt.show()
    
def plot_shap_analysis(model, X):
    shap.initjs()
    explainer = shap.TreeExplainer(model=model)
    
    if isinstance(model, lgb.LGBMRegressor):
        explainer.model.original_model.params['objective'] = 'regression'
    
    shap_values = explainer.shap_values(X)
    
    # Bar plot
    plt.figure(figsize=(20, 14))
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    plt.title("Feature Importance - Bar", fontsize=16)

    # Dot plot
    plt.figure(figsize=(20, 14))
    shap.summary_plot(shap_values, X, plot_type="dot", show=False)
    plt.title("Feature Importance - Dot", fontsize=16)

    # Adjust layout and display the plots side by side
    plt.tight_layout()
    plt.show()

    num_cols = min(3, len(X.columns))
    num_rows = (len(X.columns) + num_cols - 1) // num_cols

    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(6*num_cols, 4*num_rows))
    
    for i, ind in enumerate(X.columns):
        row = i // num_cols
        col = i % num_cols

        shap.dependence_plot(ind=ind, shap_values=shap_values, features=X, feature_names=X.columns, ax=axes[row, col], show=False)

    plt.tight_layout() 
    plt.show()
    
    n = 0
    # shap.force_plot(explainer.expected_value, shap_values[n,:], X_val.iloc[n,:])
    shap.plots._waterfall.waterfall_legacy(explainer.expected_value, shap_values[n,:], X_val.iloc[n,:])
    
    return explainer

class NullImportance:
    def __init__(self, model, cat_clos=None):
        self.model = model.__class__(**model.get_params())
        self.cat_clos = cat_clos

    def get_feature_importances(self, X, y, shuffle=False, early_stopping_rounds=100):
        name = self.model.__class__.__name__.lower()
        X_train_, X_val, y_train_, y_val = train_test_split(X, y, test_size=0.3, random_state=0)

        if shuffle:
            y_train_ = np.random.permutation(y_train_)

        if ('xgb' in name) or ('lgb' in name) or ('cat' in name):
            if ('lgb' in name) and (self.cat_clos is not None):
                self.model.fit(
                    X_train_, y_train_, eval_set=[(X_val, y_val)], categorical_feature=self.cat_clos,
                    early_stopping_rounds=early_stopping_rounds, verbose=False)
            elif ('cat' in name) and (self.cat_clos is not None):
                self.model.fit(
                    Pool(X_train_, y_train_, cat_features=self.cat_clos),
                    eval_set=Pool(X_val, y_val, cat_features=self.cat_clos),
                    early_stopping_rounds=early_stopping_rounds, verbose=False)
            else:
                self.model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)],
                               early_stopping_rounds=early_stopping_rounds, verbose=False)

        imp_df = pd.DataFrame()
        imp_df["feature"] = X.columns
        imp_df["importance"] = self.model.feature_importances_
        return imp_df.sort_values("importance", ascending=False)

    def get_null_feature_importances(self, X, y, n_repeats=10):
        self.actual_imp_df = self.get_feature_importances(X, y, shuffle=False)
        null_imp_df = pd.DataFrame()
        for i in range(n_repeats):
            imp_df = self.get_feature_importances(X, y, shuffle=True)
            imp_df["run"] = i + 1
            null_imp_df = pd.concat([null_imp_df, imp_df])
        self.null_imp_df = null_imp_df.copy()
    
    def display_distributions(self, feature, ax):
        actual_imp = self.actual_imp_df.loc[self.actual_imp_df['feature'] == feature, 'importance'].mean()
        null_imp = self.null_imp_df.loc[self.null_imp_df['feature'] == feature, 'importance']
        
        ax.hist(null_imp, label="Null importances", alpha=0.7, color='lightgrey')
        ax.axvline(x=actual_imp, color='r', linewidth=2, label='Real Target')
        ax.legend(loc="upper right")
        ax.set_title(f"Importance of {feature.upper()}", fontweight='bold')
        ax.set_xlabel(f"Distribution for {feature.upper()}")
        ax.set_ylabel("Importance")

    def plot_top_features(self, X, y, n_repeats=10, num_features=100, num_cols=5):
        self.get_null_feature_importances(X, y, n_repeats=10)
        
        top_features = self.actual_imp_df["feature"][:num_features]
        num_rows = (len(top_features) + num_cols - 1) // num_cols

        fig, axes = plt.subplots(num_rows, num_cols, figsize=(20, 4 * num_rows))
        axes = axes.flatten()

        for i, feature in enumerate(top_features):
            ax = axes[i]
            self.display_distributions(feature, ax)
        
        if len(top_features) < num_rows * num_cols:
            for j in range(len(top_features), num_rows * num_cols):
                axes[j].axis('off')
        
        plt.suptitle('Distribution of Null Importances for Top Features', fontweight='bold', fontsize=18)
        plt.tight_layout()
        plt.show()

### Xgboost

In [ ]:
NullImportance(trained_models['xgb_reg'][-1], cat_clos=categorical_columns).plot_top_features(X_val, y_val, n_repeats=20)
plot_permutation_importance(trained_models['xgb_reg'][-1], X_val, y_val)
explainer = plot_shap_analysis(trained_models['xgb_reg'][-1], X_val)

### LightGBM

In [ ]:
NullImportance(trained_models['lgb_reg'][-1], cat_clos=categorical_columns).plot_top_features(X_val, y_val, n_repeats=20)
plot_permutation_importance(trained_models['lgb_reg'][-1], X_val, y_val)
explainer = plot_shap_analysis(trained_models['lgb_reg'][-1], X_val)

### Catboost

In [ ]:
NullImportance(trained_models['cat_reg'][-1], cat_clos=categorical_columns).plot_top_features(X_val, y_val, n_repeats=20)
plot_permutation_importance(trained_models['cat_reg'][-1], X_val, y_val)
explainer = plot_shap_analysis(trained_models['cat_reg'][-1], X_val)

## Make Submission

In [ ]:
sub = pd.read_csv(os.path.join(filepath, 'sample_submission.csv'))
sub[f'{target_col}'] = test_predss
sub.to_csv('submission.csv', index=False)
sub

In [ ]:
sns.histplot(sub[f'{target_col}'])